# 01 — Prepare the hotel booking data

This notebook explains the cleaning choices and prepares the data in memory for the later modeling notebooks. The raw file is never changed, and the prepared DataFrame keeps all 119,390 rows. No modified data file is created.

## Project assumptions and preparation choices

This project studies booking-time cancellation scores for a hypothetical hotel team with capacity for 150 outreach attempts per week. The analysis uses the hotel booking demand dataset and excludes known outcome and post-booking fields from the model inputs.

The leakage exclusions, 180-day population scope, complete-cohort rule, and 60/13/13 weekly split are analytical choices explained below. They define the conditions under which the later model and policy comparisons are interpreted.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "hotel_bookings.csv").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "modeling"))

from prepare_data import (
    LEAKAGE_COLUMNS, RAW_DATA_PATH, complete_180_rows,
    model_features, prepare_data,
)

pd.set_option("display.max_columns", 60)

## 1. Load the raw data and check the target

The file is read once and copied in memory. The original DataFrame is compared with that copy after preparation to confirm it was not changed.

The target is `is_canceled`. It contains only 0 and 1. A literal check against `reservation_status == 'Canceled'` finds 1,207 exceptions: those rows are `No-Show`, and all have `is_canceled = 1`. Therefore I keep the target unchanged and validate it against the two cancellation outcomes, `Canceled` and `No-Show`. Status fields remain excluded from model features.

In [2]:
raw = pd.read_csv(RAW_DATA_PATH)
raw_before_preparation = raw.copy(deep=True)

literal_canceled = raw["reservation_status"].eq("Canceled").astype(int)
canceled_or_no_show = raw["reservation_status"].isin(
    ["Canceled", "No-Show"]
).astype(int)
display(pd.crosstab(raw["reservation_status"], raw["is_canceled"], margins=True))
print("Source rows:", f"{len(raw):,}")
print("Target values:", sorted(raw["is_canceled"].astype(int).unique().tolist()))
print("Mismatches with literal status == 'Canceled':", raw["is_canceled"].ne(literal_canceled).sum())
print("Mismatches when No-Show is also treated as canceled:", raw["is_canceled"].ne(canceled_or_no_show).sum())

is_canceled,0,1,All
reservation_status,,,
Canceled,0,43017,43017
Check-Out,75166,0,75166
No-Show,0,1207,1207
All,75166,44224,119390


Source rows: 119,390
Target values: [0, 1]
Mismatches with literal status == 'Canceled': 1207
Mismatches when No-Show is also treated as canceled: 0


## 2. Prepare a copy and audit duplicates

`source_row_id` is the zero-based row position in the source file. It is useful for joins and queue tracking, but **it is not a real booking or reservation ID**. A real outreach system would need the actual reservation key.

Exact duplicates are rows with matching values across all 32 original columns. They are flagged rather than deleted because the file has no booking ID. The code below stores every matching row in `exact_duplicate_rows` and displays the first 20 so the notebook stays readable.

In [3]:
prepared = prepare_data(raw)
print("Original DataFrame unchanged:", raw.equals(raw_before_preparation))

duplicate_mask = raw.duplicated(keep=False)
duplicate_columns = ["source_row_id", *raw.columns]
exact_duplicate_rows = (
    prepared.loc[duplicate_mask, duplicate_columns]
    .sort_values(list(raw.columns), kind="stable")
)

print(f"Exact duplicate rows flagged: {len(exact_duplicate_rows):,}")
exact_duplicate_rows.head(20)

Original DataFrame unchanged: True
Exact duplicate rows flagged: 40,165


,source_row_id,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
40772,40772,City Hotel,0,0,2015,August,32,7,0,2,2,0,0,BB,PRT,Direct,Direct,0,0,0,A,A,0,No Deposit,14,NO_COMPANY,0,Transient,75.0,0,1,Check-Out,2015-08-09
40802,40802,City Hotel,0,0,2015,August,32,7,0,2,2,0,0,BB,PRT,Direct,Direct,0,0,0,A,A,0,No Deposit,14,NO_COMPANY,0,Transient,75.0,0,1,Check-Out,2015-08-09
40821,40821,City Hotel,0,0,2015,August,32,8,0,1,2,0,0,SC,PRT,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9,NO_COMPANY,0,Transient,89.0,0,1,Check-Out,2015-08-09
40838,40838,City Hotel,0,0,2015,August,32,8,0,1,2,0,0,SC,PRT,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9,NO_COMPANY,0,Transient,89.0,0,1,Check-Out,2015-08-09
76792,76792,City Hotel,0,0,2015,August,33,10,1,0,2,0,0,SC,FRA,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,NO_AGENT,NO_COMPANY,0,Transient,75.0,0,0,Check-Out,2015-08-11
76793,76793,City Hotel,0,0,2015,August,33,10,1,0,2,0,0,SC,FRA,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,NO_AGENT,NO_COMPANY,0,Transient,75.0,0,0,Check-Out,2015-08-11
76794,76794,City Hotel,0,0,2015,August,33,10,1,0,2,0,0,SC,FRA,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,NO_AGENT,NO_COMPANY,0,Transient,75.0,0,0,Check-Out,2015-08-11
41067,41067,City Hotel,0,0,2015,August,33,11,0,1,1,0,0,BB,PRT,Corporate,Corporate,0,0,0,A,A,0,No Deposit,NO_AGENT,38,0,Transient-Party,88.0,0,0,Check-Out,2015-08-12
41073,41073,City Hotel,0,0,2015,August,33,11,0,1,1,0,0,BB,PRT,Corporate,Corporate,0,0,0,A,A,0,No Deposit,NO_AGENT,38,0,Transient-Party,88.0,0,0,Check-Out,2015-08-12
41071,41071,City Hotel,0,0,2015,August,33,11,0,1,2,0,0,BB,PRT,Direct,Direct,0,0,0,A,A,0,No Deposit,NO_AGENT,NO_COMPANY,0,Transient,80.0,0,0,Check-Out,2015-08-12


## 3. Handle missing values and categories

- Missing `children` becomes 0, with `children_was_missing` preserving that information.
- Missing country, agent, and company use explicit missing-value labels.
- Agent and company codes become strings because they are labels, not quantities.
- Text whitespace is normalized, but ambiguous categories are not guessed or combined.
- The source [data dictionary](https://doi.org/10.1016/j.dib.2018.11.126) defines meal values `Undefined` and `SC` together as “no meal package,” so that one merge is supported. `meal_was_undefined` records which rows changed. `Undefined` in other fields remains separate.

In [4]:
display(
    pd.DataFrame(
        {
            "raw missing": [
                raw["children"].isna().sum(), raw["country"].isna().sum(),
                raw["agent"].isna().sum(), raw["company"].isna().sum(),
            ],
            "prepared value": ["0 + flag", "UNKNOWN_COUNTRY", "NO_AGENT", "NO_COMPANY"],
        },
        index=["children", "country", "agent", "company"],
    )
)

print("Meal rows changed from Undefined to SC:", prepared["meal_was_undefined"].sum())
print("SC meal rows after cleaning:", prepared["meal"].eq("SC").sum())
print("Agent dtype:", prepared["agent"].dtype, "| Company dtype:", prepared["company"].dtype)

,raw missing,prepared value
children,4,0 + flag
country,488,UNKNOWN_COUNTRY
agent,16340,NO_AGENT
company,112593,NO_COMPANY


Meal rows changed from Undefined to SC: 1169
SC meal rows after cleaning: 11819
Agent dtype: string | Company dtype: string


## 4. Create dates and audit unusual values

`arrival_date` combines the three arrival fields. `booking_date_proxy = arrival_date - lead_time` is only an approximation because the file does not contain the actual booking timestamp. It is suitable for daily and weekly cohorts, not exact event ordering. `reservation_status_date` is parsed as the date when the final status was recorded. It is retained for auditing and is never a model input.

`total_nights` and `total_guests` are audit fields; the original components remain the model inputs. Negative ADR, zero-night, and zero-guest rows are flagged rather than repaired or removed.

In [5]:
display(
    pd.Series(
        {
            "arrival date start": prepared["arrival_date"].min().date(),
            "arrival date end": prepared["arrival_date"].max().date(),
            "booking proxy start": prepared["booking_date_proxy"].min().date(),
            "booking proxy end": prepared["booking_date_proxy"].max().date(),
        },
        name="date",
    ).to_frame()
)

flag_columns = [
    "negative_adr_flag",
    "zero_night_stay_flag",
    "zero_guest_booking_flag",
]
prepared[flag_columns].sum().rename("flagged rows").to_frame()

,date
arrival date start,2015-07-01
arrival date end,2017-08-31
booking proxy start,2013-06-24
booking proxy end,2017-08-31


,flagged rows
negative_adr_flag,1
zero_night_stay_flag,715
zero_guest_booking_flag,180


## 5. Keep model inputs clean and define the modeling population

The prepared table keeps outcome fields for auditing, but `CANDIDATE_MODEL_FEATURES` excludes the target and the identified leakage fields. Audit flags, row IDs, constructed dates, and the split label are also not model inputs. The remaining fields are candidate features assumed to be available at booking time; the source [data paper](https://doi.org/10.1016/j.dib.2018.11.126) notes that booking attributes can change before arrival, so this file cannot prove that every value is a true booking-time snapshot. That limitation should be stated with the model results.

The modeling population keeps bookings with at most 180 days of lead time and uses only complete Monday–Sunday booking weeks. This avoids partial booking cohorts caused by the file being bounded by arrival date (2015-07-01 to 2017-08-31).

The 180-day limit is a **scope choice, not data cleaning**. All 119,390 rows remain in the in-memory `prepared` DataFrame, but longer-lead and incomplete boundary rows are outside the modeling population. Model conclusions therefore apply to bookings made at most 180 days before arrival.

In [6]:
candidate_inputs = model_features(prepared)

display(
    pd.DataFrame(
        {"leakage field": LEAKAGE_COLUMNS, "present in model inputs": [c in candidate_inputs for c in LEAKAGE_COLUMNS]}
    )
)

,leakage field,present in model inputs
0,is_canceled,False
1,reservation_status,False
2,reservation_status_date,False
3,assigned_room_type,False
4,booking_changes,False
5,days_in_waiting_list,False


## 6. Create chronological booking periods

There is no random split. The 86 complete booking weeks are divided into 60 training weeks, 13 validation weeks, and 13 holdout weeks. This keeps every week intact and gives validation and holdout one quarter each. Rows outside the complete 180-day population are labeled `out_of_scope`.

In [7]:
modeling_population = complete_180_rows(prepared)
split_summary = (
    modeling_population.groupby("booking_period_split")
    .agg(
        rows=("source_row_id", "size"),
        weeks=("booking_week_start", "nunique"),
        first_week=("booking_week_start", "min"),
        last_week=("booking_week_start", "max"),
    )
    .reindex(["train", "validation", "holdout"])
)
split_summary

,rows,weeks,first_week,last_week
booking_period_split,,,,
train,48738,60,2015-07-06,2016-08-22
validation,11050,13,2016-08-29,2016-11-21
holdout,17075,13,2016-11-28,2017-02-20


## Suggestions for the next notebooks

- **Use the fixed chronological split.** Model on the complete 180-day population using 60 training weeks, 13 validation weeks, and 13 holdout weeks. Do not change these periods after reviewing results.
- **Keep conclusions within scope.** The primary model does not cover bookings made more than 180 days before arrival or incomplete boundary cohorts.
- **Retain the duplicate limitation.** Exact matches may be separate anonymized bookings because the file has no booking ID. Keep the flagged rows and do not claim that they are confirmed duplicate reservations.
- **Evaluate the standing queue.** Each Monday, rank active, uncalled bookings and select up to 150. Keep the booking-time cancellation probability fixed, and combine it with booking value in the ranking rather than re-scoring the booking each week.
- **Use only training labels already known by validation start.** Within the training booking period, fit later models only on rows whose arrival date is before the validation cutoff. This avoids using outcomes that would not yet have been known.
- **Separate prediction from intervention impact.** This data can estimate cancellation risk, but it cannot show whether a call and $50 credit prevent a cancellation. Return estimates need explicit assumptions or, preferably, a randomized pilot.
- **Keep the unusual-value limitation visible.** The data contains one negative ADR, 715 zero-night stays, 180 zero-guest rows, and a maximum ADR of 5,400. They remain flagged and retained because the source provides no defensible repair or exclusion rule.
- **Be careful with sparse categories.** `company` is `NO_COMPANY` for more than 94% of rows. Treat agent/company as categories and handle unseen values inside each chronological training window.